# 04 — Model Training

Train XGBoost, LightGBM, Random Forest, and Logistic Regression models
for three targets: 1X2, Over/Under 2.5, and GG/NG.

Uses temporal train/test split (no data leakage) and saves fitted models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from mls_predictor.data_loader import load_raw_data
from mls_predictor.elo import compute_elo_history
from mls_predictor.feature_engine import build_all_features, get_feature_columns
from mls_predictor.model_utils import (
    temporal_train_test_split, prepare_features,
    train_all_models, evaluate_models, build_ensemble_predictions,
    save_model, save_artifacts, calibrate_model,
)

In [ ]:
%%time
# Load and featurize
df = load_raw_data()
df = compute_elo_history(df)
df_feat = build_all_features(df)
feature_cols = get_feature_columns()['all']
print(f"Dataset: {len(df_feat)} matches, {len(feature_cols)} features")

In [ ]:
# ═══════════════════════════════════════
#  Target: 1X2 (Home/Draw/Away)
# ═══════════════════════════════════════
split_1x2 = temporal_train_test_split(df_feat, 'target_1x2', feature_cols)
X_train_1x2, X_test_1x2, imp_1x2, scl_1x2 = prepare_features(split_1x2['X_train'], split_1x2['X_test'])

models_1x2 = train_all_models(X_train_1x2, split_1x2['y_train'].values, '1x2')
eval_1x2 = evaluate_models(models_1x2, X_test_1x2, split_1x2['y_test'].values, '1x2')

# Ensemble
ens_probs = build_ensemble_predictions(models_1x2, X_test_1x2)
ens_pred = np.argmax(ens_probs, axis=1)
from sklearn.metrics import accuracy_score, log_loss
print(f"\n🏆 Ensemble — Acc: {accuracy_score(split_1x2['y_test'], ens_pred):.4f} | "
      f"LogLoss: {log_loss(split_1x2['y_test'], ens_probs, labels=[0,1,2]):.4f}")

eval_1x2

In [ ]:
# ═══════════════════════════════════════
#  Target: Over/Under 2.5
# ═══════════════════════════════════════
split_ou = temporal_train_test_split(df_feat, 'target_ou25', feature_cols)
X_train_ou, X_test_ou, imp_ou, scl_ou = prepare_features(split_ou['X_train'], split_ou['X_test'])

models_ou = train_all_models(X_train_ou, split_ou['y_train'].values, 'ou25')
eval_ou = evaluate_models(models_ou, X_test_ou, split_ou['y_test'].values, 'ou25')
eval_ou

In [ ]:
# ═══════════════════════════════════════
#  Target: GG/NG (Both Teams Score)
# ═══════════════════════════════════════
split_gg = temporal_train_test_split(df_feat, 'target_ggng', feature_cols)
X_train_gg, X_test_gg, imp_gg, scl_gg = prepare_features(split_gg['X_train'], split_gg['X_test'])

models_gg = train_all_models(X_train_gg, split_gg['y_train'].values, 'ggng')
eval_gg = evaluate_models(models_gg, X_test_gg, split_gg['y_test'].values, 'ggng')
eval_gg

In [ ]:
# ═══════════════════════════════════════
#  Save all models and preprocessing
# ═══════════════════════════════════════
for target, models, imp, scl, split in [
    ('1x2', models_1x2, imp_1x2, scl_1x2, split_1x2),
    ('ou25', models_ou, imp_ou, scl_ou, split_ou),
    ('ggng', models_gg, imp_gg, scl_gg, split_gg),
]:
    for name, model in models.items():
        save_model(model, name, target)
    save_artifacts(imp, scl, split['feature_cols'], target)

print('\n✅ All models and preprocessing artifacts saved to models/')

In [ ]:
# Combined results table
all_results = pd.concat([eval_1x2, eval_ou, eval_gg], ignore_index=True)
all_results['accuracy'] = all_results['accuracy'].apply(lambda x: f"{x*100:.1f}%")
all_results['log_loss'] = all_results['log_loss'].apply(lambda x: f"{x:.4f}")
all_results.columns = ['Model', 'Target', 'Accuracy', 'Log-Loss']
all_results